# 01 — Database Setup & Schema Exploration

## What This Notebook Does
Connect to the loan default SQLite database using PySpark,
explore the schema, understand table relationships, and
write the first credit risk SQL queries.

## What Is New vs CLV Project
CLV had one target variable (clv_next_12months).
This project has TWO:
  1. is_defaulted (0/1) — binary classification
  2. expected_loss (INR) — regression (PD x LGD x EAD)

CLV had transaction data showing spending behavior.
This project has REPAYMENT data showing payment behavior —
completely different signal, completely different features.

## The Basel III Expected Loss Formula
Expected Loss = PD x LGD x EAD

PD  = Probability of Default (what our model predicts)
LGD = Loss Given Default (% of loan lost after recovery)
EAD = Exposure at Default (outstanding amount when default occurs)

This formula is used by every bank in the world for
regulatory capital allocation under Basel III / RBI guidelines.

## New Concept: Reject Inference
27,831 loans were approved. 18,842 were rejected.
We only know the default outcome for approved loans.
Did we reject the right people? This is called the
reject inference problem — a fundamental challenge in
credit risk modeling that does not exist in CLV.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DateType, IntegerType, DoubleType
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
import urllib.request
from pathlib import Path
warnings.filterwarnings("ignore")

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"]        = r"C:\hadoop\bin;" + os.environ["PATH"]

JAR_DIR = Path("../jars").resolve()
JAR_DIR.mkdir(parents=True, exist_ok=True)
JAR_PATH = JAR_DIR / "sqlite-jdbc-3.45.1.0.jar"
JAR_URL = (
    "https://repo1.maven.org/maven2/org/xerial/sqlite-jdbc/3.45.1.0/"
    "sqlite-jdbc-3.45.1.0.jar"
)

if not JAR_PATH.exists():
    print(f"SQLite JDBC jar missing at {JAR_PATH}. Downloading...")
    urllib.request.urlretrieve(JAR_URL, str(JAR_PATH))
    print("SQLite JDBC jar downloaded successfully ✅")

DB_PATH  = os.path.abspath("../data/loan_default.db")
DB_URL   = f"jdbc:sqlite:{DB_PATH}"

LOCAL_TEMP = os.path.abspath("../data/spark_temp")
os.makedirs(LOCAL_TEMP, exist_ok=True)

spark = (
    SparkSession.builder
    .appName("LoanDefault_Setup")
    .master("local[2]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.jars", str(JAR_PATH))
    .config("spark.driver.extraClassPath", str(JAR_PATH))
    .config("spark.executor.extraClassPath", str(JAR_PATH))
    .config("spark.local.dir", LOCAL_TEMP)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version  : {spark.version}")
print(f"DB path        : {DB_PATH}")
print(f"DB exists      : {os.path.exists(DB_PATH)}")
print(f"DB size        : {os.path.getsize(DB_PATH)/1024/1024:.1f} MB")
print("SparkSession ready ✅")

Spark version  : 4.2.0
DB path        : c:\Users\dsp96\Desktop\realworld-ds-ml\finance\loan_default_risk\data\loan_default.db
DB exists      : True
DB size        : 132.0 MB
SparkSession ready ✅


In [2]:
import urllib.request
from pathlib import Path

JAR_DIR = Path("../jars").resolve()
JAR_DIR.mkdir(parents=True, exist_ok=True)
JAR_PATH = JAR_DIR / "sqlite-jdbc-3.45.1.0.jar"
JAR_URL = (
    "https://repo1.maven.org/maven2/org/xerial/sqlite-jdbc/3.45.1.0/"
    "sqlite-jdbc-3.45.1.0.jar"
)

if not JAR_PATH.exists():
    print(f"SQLite JDBC jar missing at {JAR_PATH}. Downloading...")
    urllib.request.urlretrieve(JAR_URL, str(JAR_PATH))
    print("SQLite JDBC jar downloaded successfully ✅")


def read_table(name):
    return (
        spark.read.format("jdbc")
        .option("url", DB_URL)
        .option("dbtable", name)
        .option("driver", "org.sqlite.JDBC")
        .load()
    )

profiles  = read_table("applicant_profile")
loans     = read_table("loan_applications")
repayment = read_table("repayment_history")
labels    = read_table("default_labels")

# Register as SQL views immediately
profiles.createOrReplaceTempView("applicant_profile")
loans.createOrReplaceTempView("loan_applications")
repayment.createOrReplaceTempView("repayment_history")
labels.createOrReplaceTempView("default_labels")

print("All tables loaded and registered ✅\n")
print(f"{'Table':<25} {'Rows':>10} {'Cols':>6}")
print("-" * 44)
for name, df in [
    ("applicant_profile",  profiles),
    ("loan_applications",  loans),
    ("repayment_history",  repayment),
    ("default_labels",     labels),
]:
    print(f"  {name:<23} {df.count():>10,} {len(df.columns):>6}")

All tables loaded and registered ✅

Table                           Rows   Cols
--------------------------------------------
  applicant_profile           50,250     17
  loan_applications           50,000     14
  repayment_history          813,692     12
  default_labels              27,831     11
